In [1]:
import ipaddress
from collections import deque, defaultdict

def enumerate_hosts(subnet_cidr):
    """
    Given a subnet in CIDR form (e.g., "192.168.1.0/28"),
    returns a list of host IP strings (excluding network & broadcast).
    """
    net = ipaddress.ip_network(subnet_cidr, strict=False)
    hosts = [str(ip) for ip in net.hosts()]
    return hosts

def build_broadcast_tree(hosts, root):
    """
    Build a broadcast tree over the list of hosts, rooted at `root`.
    Returns a dict map: parent_map[child] = parent (root has parent None).
    For simplicity, uses BFS style: root -> first child, root -> second child, then children of root get their children, etc.
    """
    parent_map = {h: None for h in hosts}
    visited = set()
    queue = deque()
    visited.add(root)
    queue.append(root)
    
    # Prepare iterator of other hosts
    others = [h for h in hosts if h != root]
    it = iter(others)
    
    while queue:
        current = queue.popleft()
        try:
            # assign one or more children to current
            # for simplicity: assign a fixed number, say 2 children per node, if any left
            for _ in range(2):  # you can tune this branching factor
                child = next(it)
                parent_map[child] = current
                visited.add(child)
                queue.append(child)
        except StopIteration:
            break
    
    return parent_map

def print_tree(parent_map, root):
    """
    Pretty‐print the tree structure from root.
    """
    children_map = defaultdict(list)
    for node, parent in parent_map.items():
        if parent is not None:
            children_map[parent].append(node)
    
    def _print(node, indent=0):
        print(" " * (indent*2) + node)
        for c in children_map.get(node, []):
            _print(c, indent+1)
    
    _print(root)

def broadcast(parent_map, root, message):
    """
    Simulate broadcast: root receives the message first, then forwards to its children, recursively.
    """
    children_map = defaultdict(list)
    for node, parent in parent_map.items():
        if parent is not None:
            children_map[parent].append(node)
    
    def _recv(node):
        print(f"[{node}] received message: {message}")
        for c in children_map.get(node, []):
            _recv(c)
    
    _recv(root)

def main():
    subnet = input("Enter subnet (e.g., 192.168.1.0/28): ").strip()
    try:
        hosts = enumerate_hosts(subnet)
    except Exception as e:
        print("Error parsing subnet:", e)
        return
    
    if not hosts:
        print("No hosts found in the subnet.")
        return
    
    print("Hosts in subnet:")
    for idx, h in enumerate(hosts):
        print(f"  {idx}: {h}")
    
    root_idx = input(f"Choose root host index (0–{len(hosts)-1}): ").strip()
    try:
        root_idx = int(root_idx)
        if root_idx < 0 or root_idx >= len(hosts):
            print("Index out of range.")
            return
    except:
        print("Invalid index.")
        return
    
    root = hosts[root_idx]
    print("Selected root:", root)
    
    parent_map = build_broadcast_tree(hosts, root)
    
    print("\nBroadcast tree structure:")
    print_tree(parent_map, root)
    
    print("\nSimulating broadcast:")
    broadcast(parent_map, root, "Hello, all hosts!")
    
if __name__ == "__main__":
    main()


Enter subnet (e.g., 192.168.1.0/28):  192.168.1.0/28


Hosts in subnet:
  0: 192.168.1.1
  1: 192.168.1.2
  2: 192.168.1.3
  3: 192.168.1.4
  4: 192.168.1.5
  5: 192.168.1.6
  6: 192.168.1.7
  7: 192.168.1.8
  8: 192.168.1.9
  9: 192.168.1.10
  10: 192.168.1.11
  11: 192.168.1.12
  12: 192.168.1.13
  13: 192.168.1.14


Choose root host index (0–13):  1


Selected root: 192.168.1.2

Broadcast tree structure:
192.168.1.2
  192.168.1.1
    192.168.1.4
      192.168.1.8
      192.168.1.9
    192.168.1.5
      192.168.1.10
      192.168.1.11
  192.168.1.3
    192.168.1.6
      192.168.1.12
      192.168.1.13
    192.168.1.7
      192.168.1.14

Simulating broadcast:
[192.168.1.2] received message: Hello, all hosts!
[192.168.1.1] received message: Hello, all hosts!
[192.168.1.4] received message: Hello, all hosts!
[192.168.1.8] received message: Hello, all hosts!
[192.168.1.9] received message: Hello, all hosts!
[192.168.1.5] received message: Hello, all hosts!
[192.168.1.10] received message: Hello, all hosts!
[192.168.1.11] received message: Hello, all hosts!
[192.168.1.3] received message: Hello, all hosts!
[192.168.1.6] received message: Hello, all hosts!
[192.168.1.12] received message: Hello, all hosts!
[192.168.1.13] received message: Hello, all hosts!
[192.168.1.7] received message: Hello, all hosts!
[192.168.1.14] received message: H